# KV Cache Tiering Under Long-Context Pressure

### When reuse from a slower memory tier beats recomputation

**Question.** When the reusable KV working set is larger than the device-resident cache, is it faster to reload previously computed KV from host memory or recompute the prefix?

The serving backend is treated as replaceable. The artifact of interest is the **mechanism and the measurement contract**: identical workload, controlled cache pressure, cold/warm passes, TTFT, and independent cache-hit evidence.

> **Evidence status** — the numeric results below come from a completed controlled run. The notebook keeps the recorded measurements separate from the rerun harness so a new execution cannot silently overwrite the evidence being discussed.


## 1 — Falsifiable hypothesis

Let `W` be the reusable KV working set and `G` the device-resident KV capacity.

- If `W <= G`, adding a slower tier should provide little or no benefit and may add overhead.
- If `W > G` and prefixes are reused, a host tier should win when **reload time + tier-management overhead < recomputation time**.
- A cold pass may be slower with tiering because the system must compute *and* persist KV. Therefore **cold performance is not evidence of reuse value**.

The failure condition is explicit: if warm TTFT does not improve after device-cache eviction, the tiering hypothesis fails for this workload.


## 2 — Controlled workload

| Variable | Recorded value |
|---|---:|
| Requests | 24 |
| Input length | 32,000 tokens/request |
| Output length | 64 tokens/request |
| Concurrency | 4 |
| Prompt seed | 555 |
| Passes | cold, then identical warm |
| Baseline | device-resident prefix cache only |
| Treatment | same device budget + host-backed KV tier |

The treatment changes the **cache hierarchy**, not the request set. That is the key control.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json, math

recorded = json.loads(Path("../data/kv_cache_recorded_run.json").read_text())
recorded


## 3 — Recorded result

| Configuration | Cold mean TTFT | Warm mean TTFT | Warm p99 TTFT |
|---|---:|---:|---:|
| Device cache only | 9.377 s | 9.315 s | 17.125 s |
| Device + host tier | 13.263 s | **0.336 s** | **0.845 s** |

Two details matter more than the headline speedup:

1. **Tiering made the cold pass slower**: 13.263 s vs 9.377 s. Persisting reusable state has a cost.
2. **The warm pass changed regimes**: 9.315 s → 0.336 s once previously computed KV could be reloaded after device pressure.

That is the result to explain, not merely advertise.


In [ ]:
b = recorded["device_cache_only"]
t = recorded["device_plus_host_tier"]

warm_speedup = b["warm_mean_ttft_s"] / t["warm_mean_ttft_s"]
cold_overhead = t["cold_mean_ttft_s"] - b["cold_mean_ttft_s"]
warm_saving = b["warm_mean_ttft_s"] - t["warm_mean_ttft_s"]
reuses_to_repay_cold_overhead = cold_overhead / warm_saving

print(f"warm TTFT speedup      : {warm_speedup:.2f}x")
print(f"cold tiering overhead  : {cold_overhead:.3f} s")
print(f"saving per warm reuse  : {warm_saving:.3f} s")
print(f"reuses to repay setup  : {reuses_to_repay_cold_overhead:.2f}")


### Interpretation

The recorded run repaid its cold-tier overhead in **less than one full reuse**. That does **not** imply tiering is universally beneficial. It means the tested workload had enough recomputation cost and reuse to cross the break-even point quickly.

A useful production model is:

```text
expected reuse value
    = P(reuse after eviction) × (recompute_time - reload_time)
      - persistence_overhead
      - lookup / transfer overhead
```

The right decision is workload-dependent.


## 4 — Independent evidence: requested vs hit tokens

The cache counters recorded **1,536,000 lookup-requested tokens** and **768,000 hit tokens** across the two-pass experiment. The total hit fraction is 50%, which is exactly what a cold-first / identical-warm-second design should approach when the second pass is fully reusable.

The counter is important because TTFT alone is ambiguous: a speedup could come from compilation, warm kernels, allocator state, or unrelated runtime effects. A cache-hit counter ties the performance change to the proposed mechanism.


In [ ]:
c = recorded["cache_counters"]
hit_fraction = c["lookup_hit_tokens"] / c["lookup_requested_tokens"]
print(f"requested tokens : {c['lookup_requested_tokens']:,}")
print(f"hit tokens       : {c['lookup_hit_tokens']:,}")
print(f"aggregate hit rate: {hit_fraction:.1%}")


## 5 — Second falsifier: force eviction, then ask one shared-context question

A separate test used a 30K-token shared context and a device cache that could hold ~40,960 tokens. Filler contexts were sent first to force the shared document out of the device-resident cache.

| Path | TTFT |
|---|---:|
| recompute after eviction | 2.871 s |
| reload after eviction | **0.355 s** |

The tiered path reloaded **29,952 tokens** from host memory. The measured first-token improvement was **8.1×**.

This experiment is more diagnostic than an ordinary warm-cache test because it explicitly creates the failure mode: **the reusable prefix no longer fits on device**.


In [ ]:
s = recorded["single_shared_context"]
print(f"speedup             : {s['device_only_ttft_s']/s['tiered_ttft_s']:.2f}x")
print(f"tokens reloaded     : {s['host_reloaded_tokens']:,}")
print(f"reload fraction     : {s['host_reloaded_tokens']/s['context_tokens']:.1%}")


## 6 — Capacity model

For a decoder KV cache, bytes per token are approximately:

```text
2 × layers × kv_heads × head_dim × bytes_per_element
```

The factor of two is K and V. This lets the operator reason about capacity *before* running a benchmark.


In [ ]:
def kv_bytes_per_token(layers: int, kv_heads: int, head_dim: int, bytes_per_element: int) -> int:
    return 2 * layers * kv_heads * head_dim * bytes_per_element

shape = recorded["model_shape_for_capacity_example"]
bpt = kv_bytes_per_token(**shape)
print(f"KV bytes/token : {bpt:,} ({bpt/1024:.0f} KiB)")
for gib in (8, 16, 40, 100, 150):
    tokens = int(gib * 1024**3 / bpt)
    print(f"{gib:>3} GiB holds ~{tokens:>10,} KV tokens")


## 7 — Backend-neutral TTFT harness

The experiment should not depend on a particular serving product. The adapter below measures TTFT from any streaming HTTP endpoint whose request and token parser you supply.

The benchmark contract is intentionally small:

```text
same prompts
same concurrency
same generation length
same device budget
same server warm-up policy
only cache hierarchy changes
```


In [ ]:
import time
from typing import Callable, Iterable, Any


def measure_stream_ttft(
    send_stream: Callable[[str], Iterable[Any]],
    prompt: str,
    is_token: Callable[[Any], bool],
) -> float:
    t0 = time.perf_counter()
    for event in send_stream(prompt):
        if is_token(event):
            return time.perf_counter() - t0
    raise RuntimeError("stream ended before first token")


def two_pass(prompts, send_stream, is_token):
    cold = [measure_stream_ttft(send_stream, p, is_token) for p in prompts]
    warm = [measure_stream_ttft(send_stream, p, is_token) for p in prompts]
    return {"cold": cold, "warm": warm}


## 8 — What this experiment establishes — and what it does not

**Supported by the recorded run**

- under deliberate device-cache pressure, host-tier KV reuse can reduce warm TTFT by an order of magnitude;
- the cold path can regress while the reuse path improves dramatically;
- cache-hit counters and forced-eviction tests are necessary to attribute the gain correctly.

**Not established**

- that host tiering helps when the working set already fits in device memory;
- that the same gain holds for another model, sequence length, interconnect, allocator, concurrency, or host-memory bandwidth;
- that mean TTFT alone is sufficient for production SLO decisions.

### Next experiments

1. sweep working-set / device-cache ratios from `<1` to `>4`;
2. sweep reuse probability and derive an empirical break-even surface;
3. measure host bandwidth and overlap with compute;
4. report p50/p95/p99 TTFT and throughput together;
5. test cache admission/eviction policies under mixed tenant workloads.
